# Build Streamflow Training Parquet V2

This notebook replaces the original `build_streamflow_training_parquet.ipynb` with a cleaner forecasting dataset for **multi-step discharge prediction**.

Design used here:
- One Parquet file per stream site.
- Input history: `168` hourly discharge values from the target site.
- Input context: average air temperature over the last `24` hours from climate site `2`.
- Static/current context per sample: latest snow depth from climate site `1`.
- Calendar context per sample: `year`, `month`, `day`, `hour`, `day_of_week`, `day_of_year`, `season`.
- Targets: next `24` hourly discharge values.

The output schema is intentionally fixed and model-friendly so it can be used for Temporal Fusion Transformer or other sequence forecasting models after reshaping.

## Review of Issues in the Original Notebook

The original notebook has some useful ideas, especially hourly aggregation and gap filling, but it does not match the forecasting problem you described.

Main issues:
- It builds a **next-hour** target only, not a **24-hour horizon**.
- It auto-discovers **every variable** in `public.variable` and uses all of them as features. That makes the schema unstable and harder to reproduce.
- Duplicate variable names are resolved by appending database IDs, which means feature columns can change if the database changes.
- Negative values are converted to missing values for **all variables**. That is wrong for variables like air temperature, where negative readings are physically valid.
- Snow depth is not explicitly pinned to the climate station you described, so the selected source site can drift if the availability logic changes.
- The imputation strategy is the same for all features, but snow depth is better handled as a last-observed value carried forward instead of full time interpolation.
- The notebook outputs one wide file for one target site instead of generating a standard per-site discharge forecasting dataset.
- Naming is inconsistent: the database variable is `Discharge`, but the old notebook renames it to `streamflow`. That is not fatal, but it is confusing.

This replacement notebook keeps the good parts and fixes the forecasting shape, feature selection, target construction, and variable-specific cleaning choices.

## 1. Configure Database and Dataset Settings

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine, text

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except ImportError as exc:
    raise ImportError(
        "This notebook needs PyArrow. Install it with: pip install pyarrow"
    ) from exc

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 140)

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5433")
DB_NAME = os.getenv("DB_NAME", "database")
DB_USER = os.getenv("DB_USER", "admin")
DB_PASSWORD = os.getenv("DB_PASSWORD", "password")

DATABASE_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)

TARGET_SITE_IDS = [3, 4]
DISCHARGE_SOURCE_IS_TARGET_SITE = True
AIR_TEMP_SITE_ID = 2
SNOW_DEPTH_SITE_ID = 1

LOOKBACK_HOURS = 168
AIR_TEMP_LOOKBACK_HOURS = 24
FORECAST_HORIZON_HOURS = 24
HOURLY_BINNING = "hour_end"
MAX_CONSECUTIVE_MISSING_HOURS = 10
DEFAULT_SAMPLE_STRIDE_HOURS = 6
HIGH_DENSITY_MONTHS = {3, 4, 5, 6}
HIGH_DENSITY_SAMPLE_STRIDE_HOURS = 1

OUTPUT_DIR = Path("streamflow_parquet_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SITE_OUTPUT_TEMPLATE = "discharge_training_site_{site_id}_lb168_air24avg_h24.parquet"

DISCHARGE_MATCH_TERMS = ["discharge", "streamflow", "stream flow"]
AIR_TEMP_MATCH_TERMS = ["airtemp", "air temp", "air temperature"]
SNOW_DEPTH_MATCH_TERMS = ["snowdepth", "snow depth"]



## 2. Inspect Available Sites and Variables

In [ ]:
sites_df = pd.read_sql_query(
    text("""
    SELECT site_id, site_code, name, site_type, state, county
    FROM public.site
    ORDER BY site_id
    """),
    engine,
)

variables_df = pd.read_sql_query(
    text("""
    SELECT
        v.variable_id,
        v.variable_code,
        v.name AS variable_name,
        v.variable_type,
        u.name AS unit_name,
        u.symbol AS unit_symbol
    FROM public.variable v
    LEFT JOIN public.unit u ON v.unit_id = u.unit_id
    ORDER BY v.variable_id
    """),
    engine,
)

availability_df = pd.read_sql_query(
    text("""
    SELECT
        s.site_id,
        s.site_code,
        v.variable_id,
        v.variable_code,
        v.name AS variable_name,
        COUNT(*) AS row_count,
        MIN(d.datetime_utc) AS min_datetime,
        MAX(d.datetime_utc) AS max_datetime
    FROM public.datastream d
    JOIN public.site s ON d.site_id = s.site_id
    JOIN public.variable v ON d.variable_id = v.variable_id
    GROUP BY s.site_id, s.site_code, v.variable_id, v.variable_code, v.name
    ORDER BY s.site_id, v.variable_id
    """),
    engine,
)

display(sites_df)
display(variables_df)
display(availability_df)

## 3. Helper Functions

In [ ]:
def month_to_season(month):
    if month in (12, 1, 2):
        return 0
    if month in (3, 4, 5):
        return 1
    if month in (6, 7, 8):
        return 2
    return 3


def find_variable_ids(variables, match_terms):
    searchable = (
        variables["variable_code"].fillna("") + " " + variables["variable_name"].fillna("")
    ).str.lower()
    mask = pd.Series(False, index=variables.index)
    for term in match_terms:
        mask = mask | searchable.str.contains(term.lower(), regex=False)

    matched = variables.loc[mask].copy()
    if matched.empty:
        raise ValueError(f"Could not match any variable for terms: {match_terms}")
    return matched


def assign_hourly_bucket(datetime_series, binning="hour_end"):
    if binning == "hour_end":
        return datetime_series.dt.ceil("h")
    if binning == "hour_start":
        return datetime_series.dt.floor("h")
    raise ValueError("HOURLY_BINNING must be hour_end or hour_start.")


def load_raw_feature(engine, site_id, variable_ids, feature_name, allow_negative=True):
    query = text("""
        SELECT
            d.site_id,
            d.variable_id,
            d.datetime_utc AS datetime,
            d.value
        FROM public.datastream d
        WHERE d.site_id = :site_id
          AND d.variable_id = ANY(:variable_ids)
        ORDER BY d.datetime_utc, d.variable_id
    """)
    df = pd.read_sql_query(
        query,
        engine,
        params={"site_id": int(site_id), "variable_ids": [int(v) for v in variable_ids]},
    )
    if df.empty:
        raise ValueError(f"No rows found for {feature_name} at site_id={site_id}")

    df["datetime"] = pd.to_datetime(df["datetime"])

    if not allow_negative:
        negative_count = int((df["value"] < 0).sum())
        if negative_count:
            df.loc[df["value"] < 0, "value"] = np.nan
            print(f"Converted {negative_count:,} negative {feature_name} values to missing")

    return df


def to_hourly_last_value(df, feature_name, binning="hour_end"):
    working = df.copy()
    working = working.dropna(subset=["value"]).sort_values("datetime")
    working["hourly_datetime"] = assign_hourly_bucket(working["datetime"], binning=binning)

    hourly = (
        working.groupby("hourly_datetime", as_index=False)
        .tail(1)[["hourly_datetime", "value"]]
        .rename(columns={"hourly_datetime": "datetime", "value": feature_name})
        .sort_values("datetime")
        .reset_index(drop=True)
    )
    return hourly


def add_calendar_features(df):
    out = df.copy()
    out["year"] = out["datetime"].dt.year
    out["month"] = out["datetime"].dt.month
    out["day"] = out["datetime"].dt.day
    out["hour"] = out["datetime"].dt.hour
    out["day_of_week"] = out["datetime"].dt.dayofweek
    out["day_of_year"] = out["datetime"].dt.dayofyear
    out["season"] = out["month"].map(month_to_season)
    return out


def has_consecutive_missing_gap(missing_series, min_gap_hours):
    missing = missing_series.fillna(False).astype(bool)
    run_length = 0
    for is_missing in missing:
        if is_missing:
            run_length += 1
            if run_length >= min_gap_hours:
                return True
        else:
            run_length = 0
    return False


def sample_stride_hours_for_month(month):
    if month in HIGH_DENSITY_MONTHS:
        return HIGH_DENSITY_SAMPLE_STRIDE_HOURS
    return DEFAULT_SAMPLE_STRIDE_HOURS


def write_parquet_with_pyarrow(df, output_path):
    table = pa.Table.from_pandas(df, preserve_index=False)
    pq.write_table(table, output_path)


def read_parquet_with_pyarrow(input_path):
    return pq.read_table(input_path).to_pandas()



In [ ]:
discharge_variables = find_variable_ids(variables_df, DISCHARGE_MATCH_TERMS)
air_temp_variables = find_variable_ids(variables_df, AIR_TEMP_MATCH_TERMS)
snow_depth_variables = find_variable_ids(variables_df, SNOW_DEPTH_MATCH_TERMS)

print("Discharge variable matches")
display(discharge_variables)

print("Air temperature variable matches")
display(air_temp_variables)

print("Snow depth variable matches")
display(snow_depth_variables)

## 4. Build Hourly Feature Table

In [ ]:
def build_hourly_feature_table(
    engine,
    target_site_id,
    discharge_variable_ids,
    air_temp_variable_ids,
    snow_depth_variable_ids,
    air_temp_site_id,
    snow_depth_site_id,
    hourly_binning="hour_end",
):
    discharge_raw = load_raw_feature(
        engine,
        site_id=target_site_id,
        variable_ids=discharge_variable_ids,
        feature_name="discharge",
        allow_negative=False,
    )
    air_temp_raw = load_raw_feature(
        engine,
        site_id=air_temp_site_id,
        variable_ids=air_temp_variable_ids,
        feature_name="air_temp",
        allow_negative=True,
    )
    snow_depth_raw = load_raw_feature(
        engine,
        site_id=snow_depth_site_id,
        variable_ids=snow_depth_variable_ids,
        feature_name="snow_depth",
        allow_negative=False,
    )

    discharge_hourly = to_hourly_last_value(discharge_raw, "discharge", binning=hourly_binning)
    air_temp_hourly = to_hourly_last_value(air_temp_raw, "air_temp", binning=hourly_binning)
    snow_depth_hourly = to_hourly_last_value(snow_depth_raw, "snow_depth", binning=hourly_binning)

    min_dt = max(
        discharge_hourly["datetime"].min(),
        air_temp_hourly["datetime"].min(),
        snow_depth_hourly["datetime"].min(),
    )
    max_dt = min(
        discharge_hourly["datetime"].max(),
        air_temp_hourly["datetime"].max(),
        snow_depth_hourly["datetime"].max(),
    )

    hourly_index = pd.date_range(min_dt, max_dt, freq="h")
    hourly = pd.DataFrame({"datetime": hourly_index})
    hourly["site_id"] = int(target_site_id)

    hourly = hourly.merge(discharge_hourly, on="datetime", how="left")
    hourly = hourly.merge(air_temp_hourly, on="datetime", how="left")
    hourly = hourly.merge(snow_depth_hourly, on="datetime", how="left")

    hourly = hourly.sort_values("datetime").reset_index(drop=True)
    hourly["discharge_observed"] = hourly["discharge"]
    hourly["discharge_missing_raw"] = hourly["discharge"].isna()
    hourly["air_temp_missing_raw"] = hourly["air_temp"].isna()
    hourly["snow_depth_missing_raw"] = hourly["snow_depth"].isna()

    hourly = hourly.set_index("datetime")

    hourly["discharge"] = hourly["discharge"].interpolate(method="time").ffill().bfill()
    hourly["air_temp"] = hourly["air_temp"].interpolate(method="time").ffill().bfill()
    hourly["snow_depth"] = hourly["snow_depth"].ffill().bfill()

    hourly = hourly.reset_index()
    hourly = add_calendar_features(hourly)
    return hourly



## 5. Build Supervised Samples With 24-Hour Targets

In [ ]:
def build_multihorizon_samples(
    hourly_df,
    discharge_lookback_hours=168,
    air_temp_lookback_hours=24,
    forecast_horizon_hours=24,
    max_consecutive_missing_hours=10,
):
    rows = []
    expected_step = pd.Timedelta(hours=1)
    raw_missing_columns = [
        "discharge_missing_raw",
        "air_temp_missing_raw",
        "snow_depth_missing_raw",
    ]

    hourly_df = hourly_df.sort_values(["site_id", "datetime"]).reset_index(drop=True)

    for site_id, site_df in hourly_df.groupby("site_id", sort=True):
        site_df = site_df.sort_values("datetime").reset_index(drop=True)
        last_start = len(site_df) - forecast_horizon_hours + 1
        target_start_idx = discharge_lookback_hours

        while target_start_idx < last_start:
            history_window = site_df.iloc[target_start_idx - discharge_lookback_hours:target_start_idx]
            air_window = site_df.iloc[target_start_idx - air_temp_lookback_hours:target_start_idx]
            future_window = site_df.iloc[target_start_idx:target_start_idx + forecast_horizon_hours]

            forecast_month = int(future_window.iloc[0]["month"])
            stride_hours = sample_stride_hours_for_month(forecast_month)

            full_window = site_df.iloc[
                target_start_idx - discharge_lookback_hours:target_start_idx + forecast_horizon_hours
            ]
            full_time_window = full_window["datetime"]

            if not (full_time_window.diff().dropna() == expected_step).all():
                target_start_idx += stride_hours
                continue

            if len(air_window) != air_temp_lookback_hours:
                target_start_idx += stride_hours
                continue

            if future_window["discharge_observed"].isna().any():
                target_start_idx += stride_hours
                continue

            if any(
                has_consecutive_missing_gap(full_window[column], max_consecutive_missing_hours)
                for column in raw_missing_columns
            ):
                target_start_idx += stride_hours
                continue

            sample = {
                "site_id": int(site_id),
                "history_end_time": history_window.iloc[-1]["datetime"],
                "forecast_start_time": future_window.iloc[0]["datetime"],
                "forecast_end_time": future_window.iloc[-1]["datetime"],
                "snow_depth_latest": float(history_window.iloc[-1]["snow_depth"]),
                "air_temp_avg_last_24h": float(air_window["air_temp"].mean()),
                "year": int(future_window.iloc[0]["year"]),
                "month": int(future_window.iloc[0]["month"]),
                "day": int(future_window.iloc[0]["day"]),
                "hour": int(future_window.iloc[0]["hour"]),
                "day_of_week": int(future_window.iloc[0]["day_of_week"]),
                "day_of_year": int(future_window.iloc[0]["day_of_year"]),
                "season": int(future_window.iloc[0]["season"]),
                "sample_stride_hours": int(stride_hours),
            }

            for offset in range(discharge_lookback_hours, 0, -1):
                value = history_window.iloc[discharge_lookback_hours - offset]["discharge"]
                sample[f"discharge_t-{offset}"] = float(value)

            for step in range(1, forecast_horizon_hours + 1):
                value = future_window.iloc[step - 1]["discharge_observed"]
                sample[f"target_discharge_t+{step}"] = float(value)

            rows.append(sample)
            target_start_idx += stride_hours

    return pd.DataFrame(rows)


def build_training_parquet_for_site(
    engine,
    target_site_id,
    discharge_variable_ids,
    air_temp_variable_ids,
    snow_depth_variable_ids,
    air_temp_site_id,
    snow_depth_site_id,
    output_path,
    discharge_lookback_hours=168,
    air_temp_lookback_hours=24,
    forecast_horizon_hours=24,
    hourly_binning="hour_end",
    max_consecutive_missing_hours=10,
):
    hourly = build_hourly_feature_table(
        engine=engine,
        target_site_id=target_site_id,
        discharge_variable_ids=discharge_variable_ids,
        air_temp_variable_ids=air_temp_variable_ids,
        snow_depth_variable_ids=snow_depth_variable_ids,
        air_temp_site_id=air_temp_site_id,
        snow_depth_site_id=snow_depth_site_id,
        hourly_binning=hourly_binning,
    )

    training = build_multihorizon_samples(
        hourly,
        discharge_lookback_hours=discharge_lookback_hours,
        air_temp_lookback_hours=air_temp_lookback_hours,
        forecast_horizon_hours=forecast_horizon_hours,
        max_consecutive_missing_hours=max_consecutive_missing_hours,
    )

    if training.empty:
        raise ValueError(f"No training samples created for site_id={target_site_id}")

    output_path = Path(output_path)
    write_parquet_with_pyarrow(training, output_path)
    print(f"Saved {len(training):,} rows and {training.shape[1]:,} columns to {output_path}")
    return training, hourly



## 6. Build One Parquet File Per Stream Site

In [ ]:
site_outputs = {}
site_hourly_frames = {}

for target_site_id in TARGET_SITE_IDS:
    output_path = OUTPUT_DIR / SITE_OUTPUT_TEMPLATE.format(site_id=target_site_id)
    training_df, hourly_df = build_training_parquet_for_site(
        engine=engine,
        target_site_id=target_site_id,
        discharge_variable_ids=discharge_variables["variable_id"].tolist(),
        air_temp_variable_ids=air_temp_variables["variable_id"].tolist(),
        snow_depth_variable_ids=snow_depth_variables["variable_id"].tolist(),
        air_temp_site_id=AIR_TEMP_SITE_ID,
        snow_depth_site_id=SNOW_DEPTH_SITE_ID,
        output_path=output_path,
        discharge_lookback_hours=LOOKBACK_HOURS,
        air_temp_lookback_hours=AIR_TEMP_LOOKBACK_HOURS,
        forecast_horizon_hours=FORECAST_HORIZON_HOURS,
        hourly_binning=HOURLY_BINNING,
        max_consecutive_missing_hours=MAX_CONSECUTIVE_MISSING_HOURS,
    )
    site_outputs[target_site_id] = training_df
    site_hourly_frames[target_site_id] = hourly_df

    print(f"\nPreview for site_id={target_site_id}")
    display(training_df.head())



## 7. Verify Saved Parquet Files

In [ ]:
for target_site_id in TARGET_SITE_IDS:
    output_path = OUTPUT_DIR / SITE_OUTPUT_TEMPLATE.format(site_id=target_site_id)
    saved_df = read_parquet_with_pyarrow(output_path)
    print(f"site_id={target_site_id} -> shape={saved_df.shape}")
    display(saved_df.iloc[:3, :20])

## Notes

Why this layout is a better fit:
- It matches the requested forecasting task: `168h` discharge history, one `24h` average air-temperature feature, and `24h` future discharge targets.
- It uses a fixed feature set instead of every database variable.
- It keeps negative air temperatures valid.
- It treats snow depth as a carried-forward latest state value.
- It drops any sample whose full history-plus-forecast window contains a consecutive raw-data gap of `10` hours or more in discharge, air temperature, or snow depth.
- It uses a `6h` sampling stride by default, but switches to `1h` stride for March through June to generate denser spring runoff training rows.
- It makes one file per stream site, which is cleaner for training and evaluation.

If you want a true Temporal Fusion Transformer dataset next, the next step would be to convert this wide format into a long encoder-decoder format with a sample identifier and one row per time step.

